## RunPod Setup Guide

This notebook is designed to run on [RunPod](https://www.runpod.io/) GPU pods. Follow the
steps below to get a working environment.

### 1. Choose a GPU

Pick a GPU based on which ColQwen3 precision you plan to run:

| Model | Precision | Min VRAM | Recommended RunPod GPU | On-Demand Price |
|-------|-----------|----------|------------------------|-----------------|
| **tomoro-colqwen3-embed-8b** | BF16 | ~18 GB | RTX A5000 (24 GB) | ~$0.16/hr |
| **tomoro-colqwen3-embed-8b** | BF16 | ~18 GB | RTX A6000 (48 GB) | ~$0.33/hr |
| **tomoro-ai-colqwen3-embed-8b-awq** | AWQ 4-bit | ~8 GB | RTX 4090 (24 GB) | ~$0.34/hr |

> **Budget tip:** The 24 GB RTX A5000 ($0.16/hr) is the cheapest option that fits the
> BF16 model. Embedding models need less KV-cache than generative models, so 24 GB is
> sufficient even for large batch sizes. For comfortable headroom (e.g., batch size 50+),
> use a 48 GB A6000.

### 2. Choose a Template

When creating your pod, select the **PyTorch** template:

- **Template:** `RunPod PyTorch 2.8.0` (CUDA 12.8 pre-installed)
- This gives you: Python 3.11+, PyTorch 2.8, CUDA 12.8, cuDNN, JupyterLab on port 8888

> **Why PyTorch and not plain CUDA?** vLLM depends on PyTorch for tensor operations.
> The PyTorch template has the correct CUDA toolkit, driver versions, and `torch`
> pre-installed so `pip install vllm` can link against them without version conflicts.

### 3. System Dependencies (apt)

Run the cell below **once** after pod startup to install required system packages.
These are needed by `pdf2image` (Poppler) and DOCX conversion (LibreOffice).

### 4. Python Dependencies (pip)

The second cell installs all Python packages with **pinned versions** that are tested to
work together.

In [1]:
# ── System dependencies (run once after pod startup) ───────────────
# poppler-utils  — required by pdf2image (provides pdftoppm binary)
# libreoffice    — required for DOCX-to-PDF conversion (headless mode)
# libgl1, libglib2.0-0 — image processing runtime deps
!apt-get update -qq && apt-get install -y -qq \
    poppler-utils \
    libreoffice \
    libgl1 \
    libglib2.0-0 \
    && echo "System packages installed successfully"

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
debconf: delaying package configuration, since apt-utils is not installed
Selecting previously unselected package poppler-data.
(Reading database ... 49062 files and directories currently installed.)
Preparing to unpack .../00-poppler-data_0.4.12-1_all.deb ...
Unpacking poppler-data (0.4.12-1) ...
Selecting previously unselected package libreoffice-style-colibre.
Preparing to unpack .../01-libreoffice-style-colibre_4%3a24.2.7-0ubuntu0.24.04.4_all.deb ...
Unpacking libreoffice-style-colibre (4:24.2.7-0ubuntu0.24.04.4) ...
Selecting previously unselected package libreoffice-uiconfig-common.
Preparing to unpack .../02-libreoffice-uiconfig-common_4%3a24.2.7-0ubuntu0.24.04.4_all.deb ...
Unpacking libreoffice-uiconfig-common (4:24.2.7-0ubuntu0.24.04.4) ...
Selecting previously unsele

In [2]:
# ── Python dependencies (pinned versions, tested on RunPod PyTorch 2.9 / CUDA 12.8) ──
# vLLM >= 0.11.0 is required for ColQwen3 support.
# httpx is used for HTTP requests to the vLLM server (matches production codebase).
!pip install \
    "httpx>=0.27.0" \
    "Pillow>=10.4.0,<13.0" \
    "pdf2image==1.17.0" \
    "numpy>=1.26.0" \
    "tqdm>=4.66.0" \
    "ipywidgets>=8.1.0" \
    "hf_transfer" \
    "opik"

#     "vllm>=0.16.0" \

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 24.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 27.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 44.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 39.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 70.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 18.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.0/802.0 kB 18.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 42.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.3/596.3 kB 9.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 46.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47/47 [opik]0m [opik]0m [litellm]ace-hub]


In [3]:
# Option 0 - Reduce parallelism to lower peak memory
# export MAX_JOBS=1
# !pip install "vllm @ git+https://github.com/vllm-project/vllm.git"

# Option 1 - CUDA 13.0
# !pip install https://github.com/vllm-project/vllm/releases/download/v0.16.0/vllm-0.16.0+cu130-cp38-abi3-manylinux_2_35_x86_64.whl

# Option 2 - CUDA 12.x (Recommended)
# !pip install https://github.com/vllm-project/vllm/releases/download/v0.16.0/vllm-0.16.0-cp38-abi3-manylinux_2_31_x86_64.whl

# !pip install flash-attn --no-build-isolation

!pip install -U https://wheels.vllm.ai/7e08c22b8cb65a1bea6b4bf9c52ed6e71d4acc47/vllm-0.16.1rc1.dev101%2Bg7e08c22b8-cp38-abi3-manylinux_2_31_x86_64.whl



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.7/500.7 MB 29.1 MB/s  0:00:17:00:0100:01
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 36.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 51.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 61.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 69.3 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 9.0 MB/s  0:00:30m0:00:0100:02m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 45.5 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 47.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
# ── Verify environment ─────────────────────────────────────────────
import importlib.metadata
import subprocess
import torch

print(f"Python:  {subprocess.check_output(['python3', '--version']).decode().strip()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE'}")
print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

import httpx
import numpy as np
import vllm
import PIL
print(f"vLLM:    {vllm.__version__}")
print(f"httpx:   {httpx.__version__}")
print(f"numpy:   {np.__version__}")
print(f"Pillow:  {PIL.__version__}")
print(f"pdf2image: {importlib.metadata.version('pdf2image')}")

Python:  Python 3.12.3
PyTorch: 2.10.0+cu128
CUDA:    12.8
GPU:     NVIDIA A100-SXM4-80GB
VRAM:    85.1 GB
vLLM:    0.16.1rc1.dev101+g7e08c22b8
httpx:   0.28.1
numpy:   2.1.2
Pillow:  11.0.0
pdf2image: 1.17.0


# ColQwen3 Multi-Vector Embeddings via vLLM

This notebook demonstrates how to use **TomoroAI/tomoro-colqwen3-embed-8b** via
a **vLLM HTTP server** (`/pooling` API) for multi-vector ColPali-style document
embeddings.

1. **Image embedding** — encode a single JPG/PNG into patch embeddings
2. **PDF embedding** — rasterize all pages and batch-embed them
3. **DOCX embedding** — convert to PDF, rasterize, and batch-embed
4. **Query embedding** — encode text queries for retrieval
5. **MaxSim scoring** — rank documents against queries using late-interaction scoring

### How ColPali Multi-Vector Embeddings Work

Unlike traditional dense embeddings (one vector per document), ColPali produces a **list
of patch vectors** per image or query. Each patch corresponds to a visual or textual token
region. Retrieval uses **MaxSim** (Maximum Similarity) scoring:

```
MaxSim(Q, D) = Σ_{q ∈ Q} max_{d ∈ D} cos_sim(q, d)
```

For each query patch vector, we find its maximum cosine similarity against all document
patch vectors, then sum these maxima. This late-interaction approach captures fine-grained
visual-textual alignment without requiring a cross-encoder at query time.

### Prerequisites

- **GPU**: NVIDIA GPU with ≥ 24 GB VRAM (RTX A5000 or better)
- For PDF support: `poppler-utils` system package
- For DOCX support: `libreoffice` system package

## Choosing a ColQwen3 Model

The [TomoroAI ColQwen3](https://huggingface.co/TomoroAI) collection offers embedding
models optimized for multi-vector document retrieval.

### Available Variants

| Model ID | Precision | VRAM (approx) | Embedding Dim | Best for |
|----------|-----------|---------------|---------------|----------|
| `TomoroAI/tomoro-colqwen3-embed-8b` | BF16 | ~18 GB | 320 | **This notebook** — full precision, best embedding quality |
| `TomoroAI/tomoro-ai-colqwen3-embed-8b-awq` | AWQ 4-bit | ~8 GB | 320 | Production — smaller footprint, shares GPU with other models |

> **Why full precision for this notebook?** On RunPod you typically have a dedicated GPU
> with ample VRAM. The non-AWQ BF16 model avoids quantization artifacts in embeddings,
> which matters for retrieval quality. The AWQ variant is used in the production
> `docker-compose.yml` where three models (ColPali + Qwen3-VL + Qwen2.5-7B) share a
> single A100 80GB GPU.

To switch models, change `MODEL_ID` in the configuration cell below.

### Imports and Configuration

Standard library imports plus httpx for HTTP requests, numpy for MaxSim scoring, PIL
for image handling, and tqdm for progress bars.

Configuration is split into three groups:
- **Server** — vLLM server URL, model ID, request timeout.
- **Model** — dtype, max context length, GPU memory utilization (used when starting the server).
- **Document Processing** — DPI for PDF rasterization, batch size, output directory.

In [5]:
import base64
import io
import json
import shutil
import subprocess
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path

import httpx
import numpy as np
from IPython.display import display, HTML, Image as IPImage, Markdown
from PIL import Image
from tqdm.notebook import tqdm

# ── Server Configuration ───────────────────────────────────────
VLLM_BASE_URL = "http://localhost:8004"    # vLLM server address
MODEL_ID = "TomoroAI/tomoro-colqwen3-embed-8b"
REQUEST_TIMEOUT = 120.0                     # seconds per HTTP request

# ── Model Server Parameters (used when starting vLLM) ─────────
DTYPE = "bfloat16"
MAX_MODEL_LEN = 32768
GPU_MEMORY_UTILIZATION = 0.95

# ── Document Processing ───────────────────────────────────────
PDF_DPI = 150                      # 150 DPI: good quality, 56% fewer pixels than 200
MAX_WORKERS = 8                    # concurrent HTTP requests to vLLM
EMBEDDING_DIM = 320                # ColQwen3 patch embedding dimension

OUTPUT_DIR = Path("./embedding_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Shared HTTP client for vLLM server ─────────────────────────
# A single httpx.Client instance is reused across all vLLM HTTP
# embedding functions, including the ThreadPoolExecutor workers in
# embed_pages_batch(). httpx.Client is thread-safe and maintains
# a connection pool internally, avoiding TCP setup/teardown per
# request and enabling HTTP keep-alive to the vLLM server.
if "vllm_http_client" in dir() and vllm_http_client is not None:
    vllm_http_client.close()

vllm_http_client = httpx.Client(
    timeout=REQUEST_TIMEOUT,
    limits=httpx.Limits(
        max_connections=MAX_WORKERS + 4,
        max_keepalive_connections=MAX_WORKERS,
    ),
)

print(f"Model: {MODEL_ID} ({DTYPE})")
print(f"Server: {VLLM_BASE_URL}")
print(f"Embedding dimension: {EMBEDDING_DIM}")
print(f"Max context: {MAX_MODEL_LEN}, GPU mem: {GPU_MEMORY_UTILIZATION}")
print(f"PDF DPI: {PDF_DPI}, Workers: {MAX_WORKERS}")
print(f"HTTP client: max_connections={MAX_WORKERS + 4}, keepalive={MAX_WORKERS}")

Model: TomoroAI/tomoro-colqwen3-embed-8b (bfloat16)
Server: http://localhost:8004
Embedding dimension: 320
Max context: 32768, GPU mem: 0.95
PDF DPI: 150, Workers: 8
HTTP client: max_connections=12, keepalive=8


### Opik Observability Setup

[Opik](https://github.com/comet-ml/opik) (by Comet, Apache 2.0) provides LLM tracing,
token tracking, and evaluation metrics.

Since vLLM is running as an HTTP server (not in-process), auto-instrumentation won't
work. We use **manual span creation** via `opik_client.trace()` / `trace.span()` to log
each embedding request with token counts, model name, and embedding dimensions.

All traces are sent to the **`vision-rag-colpali-embedding`** project via the
`opik_client` instance (configured with `project_name`). We intentionally avoid the
`@track` decorator because it sends traces to `Default Project` unless overridden,
and combining it with manual tracing creates duplicate entries.

**Self-hosted deployment** (one-time):
```bash
git clone https://github.com/comet-ml/opik.git && cd opik
docker compose --profile opik up -d
# UI at http://localhost:5173
```

**Cloud alternative**: Sign up at [comet.com/opik](https://www.comet.com/opik) and set
`api_key` + `workspace` below instead of `use_local=True`.

In [6]:
# ── Opik Observability ─────────────────────────────────────────
import opik

# For self-hosted local instance (Docker):
# opik.configure(use_local=True)
# opik_client = opik.Opik(project_name="vision-rag-colpali-embedding")
# print("Opik configured \u2014 traces will appear at http://localhost:5173")

# For Comet cloud (uncomment and fill in):
opik.configure(api_key="9WJfC42R22MFPi7sRdNm1pzWk", workspace="gintlakos")
opik_client = opik.Opik(project_name="vision-rag-colpali-embedding")
print("Opik configured \u2014 traces will appear at https://www.comet.com/opik")

OPIK: Opik is already configured. You can check the settings by viewing the config file at /root/.opik.config
OPIK: Configuration completed successfully. Traces will be logged to 'Default Project' project. To change the destination project, see: https://www.comet.com/docs/opik/tracing/log_traces#configuring-the-project-name


Opik configured — traces will appear at https://www.comet.com/opik


## Starting the vLLM Server

This notebook communicates with a **vLLM HTTP server** over HTTP. This mirrors the
production architecture where ColPali runs as a standalone container.

### Start in a separate terminal

Open a terminal on your RunPod pod and run:

```bash
vllm serve TomoroAI/tomoro-colqwen3-embed-8b \
    --port 8004 \
    --trust-remote-code \
    --dtype bfloat16 \
    --max-model-len 32768 \
    --gpu-memory-utilization 0.95
```

### Start from this notebook (background process)

Uncomment the `subprocess.Popen` line in the cell below to start the server in the
background. The server takes 2–3 minutes to download the model on first run (~17 GB),
then ~30–60 seconds to load into GPU memory.

### Key flags

| Flag | Purpose |
|------|---------|
| `--trust-remote-code` | Required because ColQwen3 uses custom modeling code |
| `--dtype bfloat16` | Full precision for best embedding quality |
| `--max-model-len 32768` | Maximum input context (images are tokenized into patches) |
| `--gpu-memory-utilization 0.95` | Use nearly all VRAM since this is the only model |

In [7]:
# ── Start vLLM server in the background (skip if already running) ──
server_cmd = [
    "vllm", "serve", MODEL_ID,
    "--port", "8004",
    "--trust-remote-code",
    "--dtype", DTYPE,
    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
]

print(f"Server command:")
print(f"  {' '.join(server_cmd)}")
print()
print("Uncomment the lines below to start the server from this notebook:")
print("  This may take 2-3 minutes on first run (model download)...")

# Uncomment to start:
# proc = subprocess.Popen(
#     server_cmd,
#     stdout=open("vllm_server.log", "w"),
#     stderr=subprocess.STDOUT,
# )
# print(f"\nServer started with PID: {proc.pid}")
# print("Logs: tail -f vllm_server.log")

Server command:
  vllm serve TomoroAI/tomoro-colqwen3-embed-8b --port 8004 --trust-remote-code --dtype bfloat16 --max-model-len 32768 --gpu-memory-utilization 0.95

Uncomment the lines below to start the server from this notebook:
  This may take 2-3 minutes on first run (model download)...


### Wait for Server

The cell below polls the vLLM `/health` endpoint until the server is ready. If you
started the server in a separate terminal, this will detect it immediately. If you
started it from the cell above, expect a 1–3 minute wait.

In [8]:
def wait_for_server(base_url: str = VLLM_BASE_URL, timeout: int = 300, interval: int = 5):
    """Poll the vLLM health endpoint until the server is ready."""
    start = time.time()
    while time.time() - start < timeout:
        try:
            r = httpx.get(f"{base_url}/health", timeout=5)
            if r.status_code == 200:
                elapsed = time.time() - start
                print(f"vLLM server is ready! (took {elapsed:.0f}s)")
                # Print model info
                try:
                    models = httpx.get(f"{base_url}/v1/models", timeout=5).json()
                    for m in models.get("data", []):
                        print(f"  Model: {m['id']}")
                except Exception:
                    pass
                return True
        except httpx.ConnectError:
            pass
        print(f"  Waiting for server at {base_url}... ({int(time.time() - start)}s)")
        time.sleep(interval)
    raise TimeoutError(f"vLLM server did not start within {timeout}s")


wait_for_server()

vLLM server is ready! (took 0s)
  Model: TomoroAI/tomoro-colqwen3-embed-8b


True

## Utility Functions

Eight helpers that the rest of the notebook depends on:

| Function | Purpose |
|----------|---------|
| `load_image()` | Accepts a file path, `PIL.Image`, or raw bytes and returns a normalised RGB `PIL.Image`. |
| `pdf_to_images()` | Rasterizes every page of a PDF to a PIL image using **pdf2image** (wraps Poppler). |
| `docx_to_images()` | Converts a DOCX to PDF via **LibreOffice** headless, then rasterizes the PDF. |
| `encode_image_b64()` | Encodes a PIL image to a base64 PNG string for the vLLM API. |
| `embed_images()` | Sends base64 images to `/pooling` (chat-style messages) and returns multi-vector results. |
| `embed_query()` | Sends a text query to `/pooling` and returns query patch embeddings. |
| `maxsim_score()` | Computes MaxSim between query and document multi-vector embeddings. |
| `embed_pages_batch()` | Batch-embeds all pages of a document with concurrent requests and per-page fallback. |

### `load_image` — universal image loader

In [9]:
def load_image(image_source) -> Image.Image:
    """
    Load an image from various sources and return a PIL.Image.

    Args:
        image_source: str path, Path object, PIL.Image, or bytes
    """
    if isinstance(image_source, Image.Image):
        return image_source.convert("RGB")
    elif isinstance(image_source, bytes):
        return Image.open(io.BytesIO(image_source)).convert("RGB")
    elif isinstance(image_source, (str, Path)):
        return Image.open(image_source).convert("RGB")
    else:
        raise TypeError(f"Unsupported image source type: {type(image_source)}")

### `pdf_to_images` — PDF rasterization

Uses `pdf2image.convert_from_path` which shells out to **Poppler** (`pdftoppm`).
Each page becomes a separate PIL image at the configured DPI (default 300 — high
enough for OCR-quality text, but you can lower it to 150 for faster processing).

In [10]:
def pdf_to_images(pdf_path: str | Path, dpi: int = PDF_DPI) -> list[Image.Image]:
    """
    Rasterize a PDF into a list of PIL Images (one per page).

    Requires poppler-utils system package.
    """
    from pdf2image import convert_from_path

    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    print(f"Rasterizing {pdf_path.name} at {dpi} DPI...")
    images = convert_from_path(
        str(pdf_path),
        dpi=dpi,
        fmt="jpeg",         # JPEG is 3-5x faster to encode than PNG
        jpegopt={"quality": 95},
        thread_count=4,      # parallelize Poppler rendering
    )
    print(f"  -> {len(images)} page(s) extracted")
    return images

### `docx_to_images` — DOCX conversion

DOCX files have no fixed page layout until rendered, so we rely on **LibreOffice** in
headless mode to convert the document to a PDF first, then rasterize that PDF with
`pdf_to_images()`. The two-step pipeline is:

```
DOCX  ─▶  LibreOffice --headless --convert-to pdf  ─▶  PDF  ─▶  pdf2image  ─▶  [page images]
```

The intermediate PDF is **preserved** — if you pass `save_pdf_to` (a directory path), the
converted PDF is copied there so it is not lost when the temporary directory is cleaned up.

In [11]:
def docx_to_images(
    docx_path: str | Path,
    dpi: int = PDF_DPI,
    save_pdf_to: Path | None = None,
) -> list[Image.Image]:
    """
    Convert a DOCX file to page images via LibreOffice (headless) -> PDF -> images.

    Args:
        docx_path: Path to the DOCX file.
        dpi: Resolution for PDF rasterization.
        save_pdf_to: If provided, copy the intermediate PDF to this directory
                     so it is not lost when the temp dir is cleaned up.

    Requires: libreoffice system package and poppler-utils.
    """
    docx_path = Path(docx_path).resolve()
    if not docx_path.exists():
        raise FileNotFoundError(f"DOCX not found: {docx_path}")

    with tempfile.TemporaryDirectory() as tmpdir:
        print(f"Converting {docx_path.name} to PDF via LibreOffice...")
        result = subprocess.run(
            [
                "libreoffice", "--headless", "--convert-to", "pdf",
                "--outdir", tmpdir, str(docx_path),
            ],
            capture_output=True, text=True, timeout=120,
        )

        if result.returncode != 0:
            raise RuntimeError(
                f"LibreOffice conversion failed:\n{result.stderr}\n\n"
                "Make sure LibreOffice is installed:\n"
                "  Ubuntu/Debian: sudo apt install libreoffice\n"
                "  macOS: brew install --cask libreoffice"
            )

        pdf_files = list(Path(tmpdir).glob("*.pdf"))
        if not pdf_files:
            raise RuntimeError("LibreOffice did not produce a PDF file")

        tmp_pdf = pdf_files[0]
        print(f"  -> PDF created: {tmp_pdf.name}")

        # Preserve the intermediate PDF if requested
        if save_pdf_to is not None:
            save_pdf_to = Path(save_pdf_to)
            save_pdf_to.mkdir(parents=True, exist_ok=True)
            saved_pdf = save_pdf_to / tmp_pdf.name
            shutil.copy2(tmp_pdf, saved_pdf)
            print(f"  -> PDF saved to: {saved_pdf}")

        return pdf_to_images(tmp_pdf, dpi=dpi)

### `encode_image_b64` — base64 encoding for the vLLM API

Encodes a PIL image as a base64 PNG string. The same encoded string is reused if an
image appears in multiple requests, avoiding redundant encoding.

### `embed_images` — per-image embedding via `/pooling` API

Sends images to the vLLM server's `/pooling` endpoint using **chat-style messages**
(one image per request). The image payload format:

```json
{
    "model": "TomoroAI/tomoro-colqwen3-embed-8b",
    "messages": [{
        "role": "user",
        "content": [
            {"type": "image_url", "image_url": {"url": "data:image/png;base64,<page>"}},
            {"type": "text", "text": "Describe the image."}
        ]
    }]
}
```

Each response `data` object contains a `data` field that is a **list of patch vectors**
(multi-vector), not a single vector. Each patch is a 320-dimensional float array.

### `embed_query` — text query embedding

Encodes a text string into query patch embeddings. Text queries use the simpler `input`
field format:

```json
{
    "model": "TomoroAI/tomoro-colqwen3-embed-8b",
    "input": ["user query text"]
}
```

In [12]:
def encode_image_b64(pil_image: Image.Image) -> str:
    """Encode a PIL image as a base64 PNG string (done once per image, reused across requests)."""
    buf = io.BytesIO()
    pil_image.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def embed_images(
    b64_images: list[str],
    base_url: str = VLLM_BASE_URL,
    model: str = MODEL_ID,
    timeout: float = REQUEST_TIMEOUT,
) -> list[list[list[float]]]:
    """
    Embed one or more base64-encoded images via vLLM /pooling.

    Uses the shared httpx client for connection pooling. Traces the entire
    operation as a single Opik trace with one child span per image request.

    Args:
        b64_images: List of base64-encoded PNG strings.
        base_url: vLLM server URL.
        model: Model identifier.
        timeout: Request timeout in seconds.

    Returns:
        List of multi-vector embeddings (one per image).
        Each embedding is list[list[float]] — a list of patch vectors, each 320-dim.
    """
    n_images = len(b64_images)
    trace_start = datetime.now()
    t0 = time.perf_counter()

    trace = opik_client.trace(
        name="embed_images",
        start_time=trace_start,
        input={"num_images": n_images},
        metadata={"model": model, "backend": "vllm"},
        tags=["vllm", "embedding", "image"],
    )

    embeddings = []

    try:
        for i, b64 in enumerate(b64_images):
            span_start = datetime.now()
            st0 = time.perf_counter()

            response = vllm_http_client.post(
                f"{base_url}/pooling",
                json={
                    "model": model,
                    "messages": [{
                        "role": "user",
                        "content": [
                            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                            {"type": "text", "text": "Describe the image."},
                        ],
                    }],
                },
                timeout=timeout,
            )
            response.raise_for_status()
            result = response.json()
            embedding = result["data"][0]["data"]
            embeddings.append(embedding)

            span_elapsed = time.perf_counter() - st0
            trace.span(
                name=f"embed_image_{i}",
                type="llm",
                start_time=span_start,
                end_time=datetime.now(),
                input={"image_index": i},
                output={"num_patches": len(embedding), "dim": len(embedding[0]) if embedding else 0},
                model=model,
                provider="vllm",
                metadata={"wall_clock_seconds": round(span_elapsed, 3)},
            )

        total_elapsed = time.perf_counter() - t0
        trace.end(
            end_time=datetime.now(),
            output={
                "num_embeddings": len(embeddings),
                "patches_per_image": [len(e) for e in embeddings],
                "total_seconds": round(total_elapsed, 2),
            },
        )

    except Exception as e:
        total_elapsed = time.perf_counter() - t0
        trace.end(
            end_time=datetime.now(),
            output={"error": str(e), "completed": len(embeddings), "total_seconds": round(total_elapsed, 2)},
            error_info={
                "exception_type": type(e).__name__,
                "message": str(e),
                "traceback": "",
            },
        )
        raise

    return embeddings


def embed_query(
    query: str,
    base_url: str = VLLM_BASE_URL,
    model: str = MODEL_ID,
    timeout: float = REQUEST_TIMEOUT,
) -> list[list[float]]:
    """
    Embed a text query via vLLM /pooling.

    Returns multi-vector query embeddings (list of patch vectors, each 320-dim).
    """
    trace_start = datetime.now()
    t0 = time.perf_counter()

    trace = opik_client.trace(
        name="embed_query",
        start_time=trace_start,
        input={"query": query[:500]},
        metadata={"model": model, "backend": "vllm"},
        tags=["vllm", "embedding", "query"],
    )

    try:
        response = vllm_http_client.post(
            f"{base_url}/pooling",
            json={"model": model, "input": [query]},
            timeout=timeout,
        )
        response.raise_for_status()
        data = response.json()
        embedding = data["data"][0]["data"]

        elapsed = time.perf_counter() - t0
        trace.span(
            name="vllm_embedding",
            type="llm",
            start_time=trace_start,
            end_time=datetime.now(),
            input={"query": query[:500]},
            output={"num_patches": len(embedding), "dim": len(embedding[0]) if embedding else 0},
            model=model,
            provider="vllm",
            metadata={"wall_clock_seconds": round(elapsed, 3)},
        )
        trace.end(
            end_time=datetime.now(),
            output={"num_patches": len(embedding)},
        )
        return embedding

    except Exception as e:
        elapsed = time.perf_counter() - t0
        trace.end(
            end_time=datetime.now(),
            output={"error": str(e), "total_seconds": round(elapsed, 2)},
            error_info={
                "exception_type": type(e).__name__,
                "message": str(e),
                "traceback": "",
            },
        )
        raise


def embed_queries_batch(
    queries: list[str],
    base_url: str = VLLM_BASE_URL,
    model: str = MODEL_ID,
    timeout: float = REQUEST_TIMEOUT,
) -> list[list[list[float]]]:
    """
    Embed multiple text queries in a SINGLE /pooling call.

    Dramatically faster than calling embed_query() per query because it avoids
    per-request HTTP overhead and lets vLLM batch the forward passes server-side.

    Args:
        queries: List of query strings.
        base_url: vLLM server URL.
        model: Model identifier.
        timeout: Request timeout in seconds.

    Returns:
        List of multi-vector embeddings (one per query).
    """
    trace_start = datetime.now()
    t0 = time.perf_counter()

    trace = opik_client.trace(
        name="embed_queries_batch",
        start_time=trace_start,
        input={"num_queries": len(queries), "queries": [q[:200] for q in queries]},
        metadata={"model": model, "backend": "vllm"},
        tags=["vllm", "embedding", "query"],
    )

    try:
        response = vllm_http_client.post(
            f"{base_url}/pooling",
            json={"model": model, "input": queries},
            timeout=timeout,
        )
        response.raise_for_status()
        data = response.json()
        embeddings = [item["data"] for item in data["data"]]

        elapsed = time.perf_counter() - t0
        trace.span(
            name="vllm_embedding",
            type="llm",
            start_time=trace_start,
            end_time=datetime.now(),
            input={"num_queries": len(queries)},
            output={
                "num_embeddings": len(embeddings),
                "patches_per_query": [len(e) for e in embeddings],
            },
            model=model,
            provider="vllm",
            metadata={"wall_clock_seconds": round(elapsed, 3)},
        )
        trace.end(
            end_time=datetime.now(),
            output={
                "num_embeddings": len(embeddings),
                "patches_per_query": [len(e) for e in embeddings],
                "total_seconds": round(elapsed, 2),
            },
        )
        return embeddings

    except Exception as e:
        elapsed = time.perf_counter() - t0
        trace.end(
            end_time=datetime.now(),
            output={"error": str(e), "total_seconds": round(elapsed, 2)},
            error_info={
                "exception_type": type(e).__name__,
                "message": str(e),
                "traceback": "",
            },
        )
        raise

### MaxSim Scoring — late-interaction retrieval

ColPali uses **MaxSim** (Maximum Similarity) for comparing query and document embeddings.
For each query patch vector, we find its maximum cosine similarity against all document
patch vectors, then sum these maxima:

```
MaxSim(Q, D) = Σ_{q ∈ Q} max_{d ∈ D} cos_sim(q, d)
```

This captures fine-grained alignment between query tokens and visual document patches.
The production system stores embeddings in Qdrant with `MultiVectorComparator.MAX_SIM`
(see `embedding_index.py`), but here we implement it directly in NumPy for transparency.

In [13]:
def maxsim_score(
    query_embeddings: np.ndarray | list,
    doc_embeddings: np.ndarray | list,
) -> float:
    """
    Compute MaxSim score between query and document multi-vector embeddings.

    Accepts numpy arrays (preferred) or nested Python lists.

    Args:
        query_embeddings: Query patch vectors, shape (N_q, dim).
        doc_embeddings: Document patch vectors, shape (N_d, dim).

    Returns:
        MaxSim score (higher = more relevant).
    """
    q = np.asarray(query_embeddings, dtype=np.float32)  # (N_q, dim)
    d = np.asarray(doc_embeddings, dtype=np.float32)    # (N_d, dim)

    # Normalize to unit vectors for cosine similarity
    q_norm = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-8)
    d_norm = d / (np.linalg.norm(d, axis=1, keepdims=True) + 1e-8)

    # Cosine similarity matrix: (N_q, N_d)
    sim_matrix = q_norm @ d_norm.T

    # MaxSim: for each query patch, take max similarity across all doc patches, then sum
    return float(sim_matrix.max(axis=1).sum())


def rank_documents(
    query_embeddings: np.ndarray | list,
    doc_embeddings_list: list[np.ndarray | list],
    doc_labels: list[str] | None = None,
) -> list[dict]:
    """
    Rank multiple documents against a query using MaxSim scoring.

    Args:
        query_embeddings: Query multi-vector embeddings.
        doc_embeddings_list: List of document multi-vector embeddings.
        doc_labels: Optional labels for each document.

    Returns:
        Sorted list of dicts with label, score, rank.
    """
    if doc_labels is None:
        doc_labels = [f"doc_{i}" for i in range(len(doc_embeddings_list))]

    results = []
    for label, doc_emb in zip(doc_labels, doc_embeddings_list):
        score = maxsim_score(query_embeddings, doc_emb)
        results.append({"label": label, "score": score})

    results.sort(key=lambda x: x["score"], reverse=True)
    for i, r in enumerate(results):
        r["rank"] = i + 1

    return results

### `embed_pages_batch` — concurrent per-page embedding via ThreadPoolExecutor

The central function for document embedding. Uses the same concurrent pattern as the
extraction notebook (`qwen3_vl_extract.ipynb`):

1. **Encodes** all page images to base64 (once per image).
2. **Submits** each page as an individual `/pooling` request to a
   `ThreadPoolExecutor` with `MAX_WORKERS` (default 8) concurrent threads.
3. **vLLM handles scheduling** — continuous batching and PagedAttention dynamically
   batch the in-flight requests server-side, which is more efficient than client-side
   sub-batching because the server can optimize memory allocation per-request.
4. **Error handling** — failed pages return `None` (matches production fallback).

**Why per-page concurrency?** Submitting individual pages lets vLLM's continuous batcher
decide the optimal GPU batch size dynamically, rather than the client guessing a fixed
batch size. With 8 workers keeping requests in-flight, the GPU stays saturated.

### `_embed_single_page` — single page embedding worker

Runs inside a `ThreadPoolExecutor` worker thread. Each worker:
- Creates its own `httpx.Client` (one per thread, avoids sharing)
- Sends ONE image to `/pooling` (chat-style messages format)
- Logs timing and result to Opik
- Returns a dict with `index`, `embedding`, and error info

In [14]:
def _embed_single_page(
    index: int,
    b64_image: str,
    base_url: str,
    model: str,
    timeout: float,
    trace_id: str | None = None,
) -> dict:
    """
    Embed a single page image via vLLM /pooling.

    Runs inside a ThreadPoolExecutor worker. Uses the shared vllm_http_client
    for connection pooling. If trace_id is provided, logs a child span under
    the parent trace in Opik via opik_client.span(trace_id=...).

    Args:
        index: Original page index (for result ordering).
        b64_image: Base64-encoded PNG string.
        base_url: vLLM server URL.
        model: Model identifier.
        timeout: Request timeout in seconds.
        trace_id: Parent Opik trace ID for hierarchical tracing.

    Returns:
        Dict with keys: index (int), embedding (list[list[float]] | None),
        num_patches (int), elapsed (float), error (str | None).
    """
    start_time = datetime.now()
    t0 = time.perf_counter()

    try:
        response = vllm_http_client.post(
            f"{base_url}/pooling",
            json={
                "model": model,
                "messages": [{
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_image}"}},
                        {"type": "text", "text": "Describe the image."},
                    ],
                }],
            },
            timeout=timeout,
        )
        response.raise_for_status()
        result = response.json()

        elapsed = time.perf_counter() - t0
        end_time = datetime.now()

        embedding = result["data"][0]["data"]

        # Log child span under parent trace (thread-safe: uses client queue, not shared Trace object)
        if trace_id is not None:
            opik_client.span(
                trace_id=trace_id,
                name=f"embed_page_{index}",
                type="llm",
                start_time=start_time,
                end_time=end_time,
                input={"page_index": index},
                output={
                    "num_patches": len(embedding),
                    "dim": len(embedding[0]) if embedding else 0,
                },
                model=model,
                provider="vllm",
                metadata={"wall_clock_seconds": round(elapsed, 3)},
            )

        return {
            "index": index,
            "embedding": embedding,
            "num_patches": len(embedding),
            "elapsed": elapsed,
            "error": None,
        }

    except Exception as e:
        elapsed = time.perf_counter() - t0
        end_time = datetime.now()

        # Log error span under parent trace
        if trace_id is not None:
            opik_client.span(
                trace_id=trace_id,
                name=f"embed_page_{index}",
                type="llm",
                start_time=start_time,
                end_time=end_time,
                input={"page_index": index},
                model=model,
                provider="vllm",
                error_info={
                    "exception_type": type(e).__name__,
                    "message": str(e),
                    "traceback": "",
                },
                metadata={"wall_clock_seconds": round(elapsed, 3)},
            )

        return {
            "index": index,
            "embedding": None,
            "num_patches": 0,
            "elapsed": elapsed,
            "error": str(e),
        }


def embed_pages_batch(
    page_images: list[Image.Image],
    max_workers: int = MAX_WORKERS,
) -> list[list[list[float]] | None]:
    """
    Embed all page images concurrently via vLLM /pooling.

    Creates a single parent Opik trace. Each page becomes a child span under
    that trace, providing a hierarchical view in the dashboard. Uses the shared
    httpx client for connection pooling across all workers.

    Args:
        page_images: List of PIL Images to embed.
        max_workers: Number of concurrent HTTP requests (default MAX_WORKERS=8).

    Returns:
        List of multi-vector embeddings (one per page).
        Failed pages return None (matches production fallback pattern from ingestion.py).
    """
    n_pages = len(page_images)
    batch_start_time = datetime.now()
    batch_t0 = time.perf_counter()

    # Create parent trace BEFORE work starts
    trace = opik_client.trace(
        name="embed_pages_batch",
        start_time=batch_start_time,
        input={"num_pages": n_pages, "max_workers": max_workers},
        metadata={"model": MODEL_ID, "backend": "vllm"},
        tags=["vllm", "embedding", "batch"],
    )

    # Step 1: Encode all images to base64 (CPU-bound, sequential with progress)
    encode_t0 = time.perf_counter()
    b64_images = [encode_image_b64(img) for img in tqdm(page_images, desc="Encoding images")]
    encode_elapsed = time.perf_counter() - encode_t0

    # Log encoding phase as a span
    trace.span(
        name="base64_encoding",
        type="general",
        start_time=batch_start_time,
        end_time=datetime.now(),
        input={"num_images": n_pages},
        output={"encoding_seconds": round(encode_elapsed, 2)},
        metadata={"phase": "preprocessing"},
    )

    print(f"Embedding {n_pages} page(s) with {max_workers} concurrent workers")

    # Step 2: Pre-allocate results list to maintain original page order
    results: list[list[list[float]] | None] = [None] * n_pages

    # Step 3: Submit each page as an individual request, passing trace.id for child spans
    embed_t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                _embed_single_page, i, b64, VLLM_BASE_URL, MODEL_ID, REQUEST_TIMEOUT,
                trace_id=trace.id,
            ): i
            for i, b64 in enumerate(b64_images)
        }

        with tqdm(total=n_pages, desc="Embedding pages") as pbar:
            for future in as_completed(futures):
                result = future.result()
                idx = result["index"]
                if result["error"] is None:
                    results[idx] = result["embedding"]
                    pbar.set_postfix(page=idx + 1, patches=result["num_patches"])
                else:
                    results[idx] = None
                    print(f"  Page {idx + 1}: FAILED ({result['error']})")
                pbar.update(1)

    batch_elapsed = time.perf_counter() - batch_t0
    embed_elapsed = time.perf_counter() - embed_t0
    successful = sum(1 for e in results if e is not None)
    print(f"\nEmbedding complete: {successful}/{n_pages} pages successful")

    # End the parent trace with summary output
    trace.end(
        end_time=datetime.now(),
        output={
            "successful": successful,
            "failed": n_pages - successful,
            "encoding_seconds": round(encode_elapsed, 2),
            "embedding_seconds": round(embed_elapsed, 2),
            "total_seconds": round(batch_elapsed, 2),
        },
    )

    return results

## Embedding Functions Summary

The embedding functions defined above use the vLLM HTTP server:

- `embed_images` — embed base64-encoded images via `/pooling` API
- `embed_query` — embed a text query via `/pooling` API
- `embed_queries_batch` — embed multiple text queries concurrently
- `embed_pages_batch` — embed all pages of a document concurrently via ThreadPoolExecutor

In [15]:
# ── Embedding backend confirmation ─────────────────────────────────
# All embedding functions use the vLLM HTTP server.
print(f"Backend: vLLM HTTP Server ({VLLM_BASE_URL})")
print(f"  embed_images, embed_query, embed_queries_batch, embed_pages_batch -> vLLM HTTP API")

Backend: vLLM HTTP Server (http://localhost:8004)
  embed_images, embed_query, embed_queries_batch, embed_pages_batch -> vLLM HTTP API


## Embedding Performance Profiler

Benchmarks embedding throughput using synthetic test images. Measures end-to-end
HTTP round-trip time per page via `embed_pages_batch`, including network overhead,
vLLM server scheduling, and base64 encoding.

### Interpreting Results

| If throughput is lower than expected... | Try... |
|---|---|
| Low pages/second | Increase `MAX_WORKERS` in `embed_pages_batch` |
| High latency per page | Lower `PDF_DPI` to reduce image resolution |

In [ ]:
# ── Embedding Performance Profiler ─────────────────────────────
# Benchmarks embedding throughput using synthetic test images.

import gc


def profile_embedding(
    n_pages: int = 16,
    image_size: tuple[int, int] = (1275, 1650),  # A4 at 150 DPI
    warmup_pages: int = 2,
):
    """
    Profile embedding throughput using synthetic test images.

    Creates n_pages random images and benchmarks the vLLM HTTP server backend,
    printing per-stage timing and an overall pages/second rate.

    Args:
        n_pages: Number of synthetic test images.
        image_size: (width, height) in pixels.
        warmup_pages: Number of pages to run as warmup (excluded from timing).
    """
    print(f"Creating {n_pages} synthetic test images ({image_size[0]}x{image_size[1]})...")
    test_images = [
        Image.fromarray(
            np.random.randint(0, 255, (image_size[1], image_size[0], 3), dtype=np.uint8)
        )
        for _ in range(n_pages)
    ]

    # ── Warmup ────────────────────────────────────────────────
    if warmup_pages > 0:
        print(f"\nWarming up ({warmup_pages} page(s))...")
        warmup_images = test_images[:warmup_pages]
        warmup_b64 = [encode_image_b64(img) for img in warmup_images]
        _ = embed_images(warmup_b64)
        print("  Warmup done.\n")

    # ── Benchmark ─────────────────────────────────────────────
    print(f"Benchmarking {n_pages} pages...")

    t0 = time.perf_counter()
    results = embed_pages_batch(test_images)
    wall = time.perf_counter() - t0
    per_page = wall / n_pages
    successful = sum(1 for e in results if e is not None)

    print(f"\n{'=' * 60}")
    print(f"Results: {n_pages} pages, vLLM HTTP Server")
    print(f"  Concurrent HTTP (ThreadPoolExecutor, {MAX_WORKERS} workers):")
    print(f"  Wall clock: {wall:.2f}s  →  {per_page:.3f}s/page  ({n_pages/wall:.1f} pages/s)")
    print(f"  Successful: {successful}/{n_pages}")

    print(f"\nProfile complete.")


# ── Run the profiler ───────────────────────────────────────────
profile_embedding(n_pages=16)

## Embed a Single Image (JPG / PNG)

The simplest use case: load any image file and generate its multi-vector embedding.

The result is a **list of patch vectors** — one per visual token region. The number of
patches depends on the image resolution and the model's visual tokenizer. A typical
document page at 300 DPI produces 200–800 patches, each 320-dimensional.

All output files are saved to a per-document subfolder inside `embedding_output/`.

In [16]:
# ── Configure your image path here ─────────────────────────────
IMAGE_PATH = "./test/test_png.png"  # <-- Change this to your image

if Path(IMAGE_PATH).exists():
    img_stem = Path(IMAGE_PATH).stem
    img_output_dir = OUTPUT_DIR / img_stem
    img_output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Processing: {IMAGE_PATH}")
    pil_img = load_image(IMAGE_PATH)
    print(f"  Image size: {pil_img.size[0]}x{pil_img.size[1]}")

    # Encode and embed
    b64 = encode_image_b64(pil_img)
    image_embeddings = embed_images([b64])
    image_embedding = image_embeddings[0]

    # Save image copy to output dir
    pil_img.save(img_output_dir / f"{img_stem}.png", "PNG")

    # Handle both numpy arrays and nested lists
    emb_array = np.asarray(image_embedding, dtype=np.float32)
    n_patches, n_dims = emb_array.shape

    print(f"\n{'=' * 60}")
    print(f"Image Embedding Result")
    print(f"{'=' * 60}")
    print(f"  Number of patches:   {n_patches}")
    print(f"  Embedding dimension: {n_dims}")
    print(f"  Total floats:        {n_patches * n_dims:,}")
    print(f"  Shape:               ({n_patches}, {n_dims})")
    print(f"\n  First patch (first 10 values):")
    print(f"    {emb_array[0, :10].tolist()}")
    print(f"\n  Patch norms (first 5):")
    norms = np.linalg.norm(emb_array[:5], axis=1)
    print(f"    {[round(float(n), 4) for n in norms]}")
    print(f"\nAll files saved in: {img_output_dir}")
else:
    print(f"Image not found: {IMAGE_PATH}")
    print("Please update IMAGE_PATH to point to a valid JPG or PNG file.")

OPIK: Started logging traces to the "vision-rag-colpali-embedding" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=019ca8ce-6179-7bfe-96d0-6013ca24643f&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBpLw==.


Processing: ./test/test_png.png
  Image size: 795x904

Image Embedding Result
  Number of patches:   711
  Embedding dimension: 320
  Total floats:        227,520
  Shape:               (711, 320)

  First patch (first 10 values):
    [0.045076705515384674, 0.03543153777718544, 0.029818473383784294, 0.024198567494750023, 0.0060025667771697044, -0.023664021864533424, 0.06113276258111, 0.001202244427986443, 0.00878958497196436, 0.07228224724531174]

  Patch norms (first 5):
    [1.0, 1.0, 1.0, 1.0, 1.0]

All files saved in: embedding_output/test_png


### Inspecting Multi-Vector Embeddings

Let's examine the structure of the embedding result in detail. Understanding the shape
and distribution of patch vectors is important for:

- **Storage planning**: Multi-vector embeddings are much larger than single-vector. For
  example, 500 patches × 320 dims × 4 bytes = ~625 KB per page vs ~1.3 KB for a single
  320-dim vector.
- **Retrieval quality**: More patches = finer-grained matching, but also more compute at
  search time.
- **Debugging**: Patch norm distributions reveal model behavior (are all patches equally
  informative? Are some near-zero?).

In [17]:
if "image_embedding" in dir() and image_embedding is not None:
    emb_array = np.asarray(image_embedding, dtype=np.float32)

    print(f"Embedding shape: {emb_array.shape}")
    print(f"Dtype: {emb_array.dtype}")
    print(f"Memory: {emb_array.nbytes / 1024:.1f} KB")

    # Patch norm statistics
    norms = np.linalg.norm(emb_array, axis=1)
    print(f"\nPatch vector norms:")
    print(f"  Mean:   {norms.mean():.4f}")
    print(f"  Std:    {norms.std():.4f}")
    print(f"  Min:    {norms.min():.4f}")
    print(f"  Max:    {norms.max():.4f}")

    # Value distribution
    print(f"\nValue distribution across all patches:")
    print(f"  Mean:   {emb_array.mean():.6f}")
    print(f"  Std:    {emb_array.std():.6f}")
    print(f"  Min:    {emb_array.min():.6f}")
    print(f"  Max:    {emb_array.max():.6f}")

    # Self-similarity: how diverse are the patches?
    normed = emb_array / (norms[:, None] + 1e-8)
    self_sim = normed @ normed.T
    mask = ~np.eye(len(self_sim), dtype=bool)
    print(f"\nInter-patch cosine similarity (off-diagonal):")
    print(f"  Mean: {self_sim[mask].mean():.4f}")
    print(f"  Std:  {self_sim[mask].std():.4f}")
    redundancy = "high" if self_sim[mask].mean() > 0.5 else "moderate" if self_sim[mask].mean() > 0.3 else "low"
    print(f"  This indicates {redundancy} redundancy between patches")
else:
    print("No image embedding available. Run the image embedding cell first.")

Embedding shape: (711, 320)
Dtype: float32
Memory: 888.8 KB

Patch vector norms:
  Mean:   1.0000
  Std:    0.0000
  Min:    1.0000
  Max:    1.0000

Value distribution across all patches:
  Mean:   0.000796
  Std:    0.055896
  Min:    -0.237753
  Max:    0.243691

Inter-patch cosine similarity (off-diagonal):
  Mean: 0.1806
  Std:  0.1607
  This indicates low redundancy between patches


## Embed a PDF Document

Point `PDF_PATH` at any PDF file. The pipeline:
1. **Rasterize** — `pdf_to_images()` converts every page to a PIL image at 300 DPI.
2. **Batch embed** — `embed_pages_batch()` sends all page images to vLLM via
   concurrent `/pooling` requests. Each request contains a single page image.
3. **Inspect** — View per-page patch counts and embedding dimensions.

All intermediate files are saved to a per-document subfolder:

```
embedding_output/<pdf_stem>/
    page_001.png            # rasterized page images
    page_002.png
```

In [18]:
# ── Configure your PDF path here ────────────────────────────────
PDF_PATH = "./test/test_pdf.pdf"  # <-- Change this to your PDF

if Path(PDF_PATH).exists():
    pdf_stem = Path(PDF_PATH).stem
    pdf_output_dir = OUTPUT_DIR / pdf_stem
    pdf_output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output folder: {pdf_output_dir}")

    # Rasterize
    page_images = pdf_to_images(PDF_PATH, dpi=PDF_DPI)

    # Batch embed FIRST (don't waste time saving images before embedding)
    pdf_embeddings = embed_pages_batch(page_images)

    # Save page images AFTER embedding (use JPEG for speed)
    for i, img in enumerate(page_images, 1):
        img.save(pdf_output_dir / f"page_{i:03d}.jpg", "JPEG", quality=95)
    print(f"Saved {len(page_images)} page image(s) to {pdf_output_dir}")

    print(f"\n{'=' * 60}")
    print(f"PDF Embedding Complete: {PDF_PATH}")
    print(f"{'=' * 60}")
    print(f"  Total pages:  {len(page_images)}")
    print(f"  Successful:   {sum(1 for e in pdf_embeddings if e is not None)}")
    print(f"\n  Per-page breakdown:")
    for i, emb in enumerate(pdf_embeddings, 1):
        if emb is not None:
            emb_arr = np.asarray(emb)
            print(f"    Page {i}: {emb_arr.shape[0]} patches x {emb_arr.shape[1]} dims")
        else:
            print(f"    Page {i}: FAILED")
    print(f"\nAll page images saved in: {pdf_output_dir}")
else:
    print(f"PDF not found: {PDF_PATH}")
    print("Please update PDF_PATH to point to a valid PDF file.")

Output folder: embedding_output/test_pdf
Rasterizing test_pdf.pdf at 150 DPI...
  -> 81 page(s) extracted


Encoding images:   0%|          | 0/81 [00:00<?, ?it/s]

Embedding 81 page(s) with 8 concurrent workers


Embedding pages:   0%|          | 0/81 [00:00<?, ?it/s]


Embedding complete: 81/81 pages successful
Saved 81 page image(s) to embedding_output/test_pdf

PDF Embedding Complete: ./test/test_pdf.pdf
  Total pages:  81
  Successful:   81

  Per-page breakdown:
    Page 1: 1271 patches x 320 dims
    Page 2: 1271 patches x 320 dims
    Page 3: 1271 patches x 320 dims
    Page 4: 1271 patches x 320 dims
    Page 5: 1271 patches x 320 dims
    Page 6: 1271 patches x 320 dims
    Page 7: 1271 patches x 320 dims
    Page 8: 1271 patches x 320 dims
    Page 9: 1271 patches x 320 dims
    Page 10: 1271 patches x 320 dims
    Page 11: 1271 patches x 320 dims
    Page 12: 1271 patches x 320 dims
    Page 13: 1271 patches x 320 dims
    Page 14: 1271 patches x 320 dims
    Page 15: 1271 patches x 320 dims
    Page 16: 1271 patches x 320 dims
    Page 17: 1271 patches x 320 dims
    Page 18: 1271 patches x 320 dims
    Page 19: 1271 patches x 320 dims
    Page 20: 1271 patches x 320 dims
    Page 21: 1271 patches x 320 dims
    Page 22: 1271 patches x 32

## Embed a DOCX Document

Same pipeline as PDF, but with an extra conversion step at the front:

```
DOCX  ─▶  LibreOffice (headless)  ─▶  PDF  ─▶  pdf2image  ─▶  [page images]  ─▶  embed_pages_batch()
```

All intermediate files are preserved in the output subfolder:

```
embedding_output/<docx_stem>/
    <docx_stem>.pdf          # intermediate PDF from LibreOffice (preserved)
    page_001.png             # rasterized page images
    page_002.png
```

**System requirement:** LibreOffice must be installed:
- Ubuntu/Debian: `sudo apt install libreoffice`
- macOS: `brew install --cask libreoffice`
- Windows: install from [libreoffice.org](https://www.libreoffice.org/) and ensure it is on your PATH

In [19]:
# ── Configure your DOCX path here ───────────────────────────────
DOCX_PATH = "./test/test_docx.docx"  # <-- Change this to your DOCX

if Path(DOCX_PATH).exists():
    docx_stem = Path(DOCX_PATH).stem
    docx_output_dir = OUTPUT_DIR / docx_stem
    docx_output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output folder: {docx_output_dir}")

    # Convert DOCX to images, saving the intermediate PDF
    page_images = docx_to_images(DOCX_PATH, dpi=PDF_DPI, save_pdf_to=docx_output_dir)

    # Save page images
    for i, img in enumerate(page_images, 1):
        img.save(docx_output_dir / f"page_{i:03d}.png", "PNG")
    print(f"Saved {len(page_images)} page image(s) to {docx_output_dir}")

    # Batch embed
    docx_embeddings = embed_pages_batch(page_images)

    print(f"\n{'=' * 60}")
    print(f"DOCX Embedding Complete: {DOCX_PATH}")
    print(f"{'=' * 60}")
    print(f"  Total pages:  {len(page_images)}")
    print(f"  Successful:   {sum(1 for e in docx_embeddings if e is not None)}")
    print(f"\n  Per-page breakdown:")
    for i, emb in enumerate(docx_embeddings, 1):
        if emb is not None:
            print(f"    Page {i}: {len(emb)} patches x {len(emb[0])} dims")
        else:
            print(f"    Page {i}: FAILED")
    print(f"\nAll files saved in: {docx_output_dir}")
else:
    print(f"DOCX not found: {DOCX_PATH}")
    print("Please update DOCX_PATH to point to a valid DOCX file.")
    print("Note: LibreOffice must be installed for DOCX conversion.")

Output folder: embedding_output/test_docx
Converting test_docx.docx to PDF via LibreOffice...
  -> PDF created: test_docx.pdf
  -> PDF saved to: embedding_output/test_docx/test_docx.pdf
Rasterizing test_docx.pdf at 150 DPI...
  -> 220 page(s) extracted
Saved 220 page image(s) to embedding_output/test_docx


Encoding images:   0%|          | 0/220 [00:00<?, ?it/s]

Embedding 220 page(s) with 8 concurrent workers


Embedding pages:   0%|          | 0/220 [00:00<?, ?it/s]


Embedding complete: 220/220 pages successful

DOCX Embedding Complete: ./test/test_docx.docx
  Total pages:  220
  Successful:   220

  Per-page breakdown:
    Page 1: 1251 patches x 320 dims
    Page 2: 1251 patches x 320 dims
    Page 3: 1251 patches x 320 dims
    Page 4: 1251 patches x 320 dims
    Page 5: 1251 patches x 320 dims
    Page 6: 1251 patches x 320 dims
    Page 7: 1251 patches x 320 dims
    Page 8: 1251 patches x 320 dims
    Page 9: 1251 patches x 320 dims
    Page 10: 1251 patches x 320 dims
    Page 11: 1251 patches x 320 dims
    Page 12: 1251 patches x 320 dims
    Page 13: 1251 patches x 320 dims
    Page 14: 1251 patches x 320 dims
    Page 15: 1251 patches x 320 dims
    Page 16: 1251 patches x 320 dims
    Page 17: 1251 patches x 320 dims
    Page 18: 1251 patches x 320 dims
    Page 19: 1251 patches x 320 dims
    Page 20: 1251 patches x 320 dims
    Page 21: 1251 patches x 320 dims
    Page 22: 1251 patches x 320 dims
    Page 23: 1251 patches x 320 dims
 

## Query Embedding (Text-Only)

For retrieval, you encode text queries into the same multi-vector embedding space as
document images. The query format sends plain text strings to the `/pooling` endpoint
(as opposed to the chat-style messages format used for images).

Query embeddings typically have **fewer patches** than image embeddings (since text is
shorter than a full document page), but they live in the same 320-dimensional space
and are compared using the same MaxSim scoring.

**Batching matters here.** All queries are embedded in a **single forward pass** via
`embed_queries_batch()`. On the HF backend this avoids N separate 8B-model forward
passes (each taking several seconds); on vLLM it sends one HTTP request instead of N.

This matches the production `_encode_query()` method in `orchestrator.py`.

In [20]:
# ── Configure your queries here ─────────────────────────────────
QUERIES = [
    "What is the company's revenue for Q4 2024?",
    "Show me the organizational structure diagram",
    "Find the terms and conditions section",
]

# Batch ALL queries in a single forward pass (instead of one-at-a-time loop).
# On the HF backend this is ~Nx faster because it avoids N separate 8B-model
# forward passes. On vLLM it sends one HTTP request instead of N.
print(f"Embedding {len(QUERIES)} queries in a single batch...")
query_embeddings_list = embed_queries_batch(QUERIES)

for query, q_emb in zip(QUERIES, query_embeddings_list):
    q_arr = np.asarray(q_emb, dtype=np.float32)
    print(f"\nQuery: {query!r}")
    print(f"  Patches:   {q_arr.shape[0]}")
    print(f"  Dimension: {q_arr.shape[1]}")
    print(f"  Shape:     {q_arr.shape}")

print(f"\n{'=' * 60}")
print(f"Query Embedding Summary")
print(f"{'=' * 60}")
for query, q_emb in zip(QUERIES, query_embeddings_list):
    q_arr = np.asarray(q_emb, dtype=np.float32)
    norms = np.linalg.norm(q_arr, axis=1)
    print(f"  {query[:50]:.<55s} {q_arr.shape[0]:>4d} patches, norm mean={norms.mean():.4f}")

Embedding 3 queries in a single batch...

Query: "What is the company's revenue for Q4 2024?"
  Patches:   15
  Dimension: 320
  Shape:     (15, 320)

Query: 'Show me the organizational structure diagram'
  Patches:   6
  Dimension: 320
  Shape:     (6, 320)

Query: 'Find the terms and conditions section'
  Patches:   6
  Dimension: 320
  Shape:     (6, 320)

Query Embedding Summary
  What is the company's revenue for Q4 2024?.............   15 patches, norm mean=1.0000
  Show me the organizational structure diagram...........    6 patches, norm mean=1.0000
  Find the terms and conditions section..................    6 patches, norm mean=1.0000


## Retrieval: MaxSim Scoring

Now that we have both query embeddings and document embeddings, we can rank documents
against queries using MaxSim.

This demonstrates the core retrieval mechanism used in the production pipeline
(`orchestrator.py`), where query embeddings are compared against Qdrant-stored document
embeddings using `MultiVectorComparator.MAX_SIM`. Here we compute MaxSim directly in
NumPy for transparency.

### Ranking

For each query, all available document embeddings (from the image, PDF pages, and DOCX
pages processed above) are scored and ranked. Higher MaxSim scores indicate stronger
visual-textual alignment.

In [21]:
# Collect all available document embeddings
all_doc_embeddings = []
all_doc_labels = []

# From single image
if "image_embedding" in dir() and image_embedding is not None:
    all_doc_embeddings.append(image_embedding)
    all_doc_labels.append(f"img: {Path(IMAGE_PATH).name}")

# From PDF pages
if "pdf_embeddings" in dir():
    for i, emb in enumerate(pdf_embeddings, 1):
        if emb is not None:
            all_doc_embeddings.append(emb)
            all_doc_labels.append(f"pdf: {Path(PDF_PATH).name} p.{i}")

# From DOCX pages
if "docx_embeddings" in dir():
    for i, emb in enumerate(docx_embeddings, 1):
        if emb is not None:
            all_doc_embeddings.append(emb)
            all_doc_labels.append(f"docx: {Path(DOCX_PATH).name} p.{i}")

if all_doc_embeddings and query_embeddings_list:
    print(f"Scoring {len(QUERIES)} queries against {len(all_doc_embeddings)} documents\n")

    for query, q_emb in zip(QUERIES, query_embeddings_list):
        print(f"Query: {query!r}")
        rankings = rank_documents(q_emb, all_doc_embeddings, all_doc_labels)
        for r in rankings[:5]:  # Show top 5
            print(f"  #{r['rank']:2d}  score={r['score']:8.4f}  {r['label']}")
        print()
else:
    print("No document embeddings or query embeddings available.")
    print("Run the image/PDF/DOCX embedding sections and query embedding section first.")

Scoring 3 queries against 302 documents

Query: "What is the company's revenue for Q4 2024?"
  # 1  score=  6.7460  docx: test_docx.docx p.106
  # 2  score=  5.9613  docx: test_docx.docx p.113
  # 3  score=  5.9199  docx: test_docx.docx p.190
  # 4  score=  5.9080  docx: test_docx.docx p.193
  # 5  score=  5.8620  docx: test_docx.docx p.102

Query: 'Show me the organizational structure diagram'
  # 1  score=  4.0901  docx: test_docx.docx p.18
  # 2  score=  4.0607  docx: test_docx.docx p.219
  # 3  score=  4.0394  docx: test_docx.docx p.17
  # 4  score=  3.9913  docx: test_docx.docx p.22
  # 5  score=  3.8968  docx: test_docx.docx p.16

Query: 'Find the terms and conditions section'
  # 1  score=  2.8050  pdf: test_pdf.pdf p.69
  # 2  score=  2.7473  docx: test_docx.docx p.129
  # 3  score=  2.6877  docx: test_docx.docx p.127
  # 4  score=  2.6854  docx: test_docx.docx p.11
  # 5  score=  2.6841  docx: test_docx.docx p.179



### Score Matrix

The full query-document score matrix reveals which queries match which pages.
Diagonal dominance (when queries match their intended documents) indicates
correct retrieval behavior. Cells are color-coded: darker green = higher relevance.

In [22]:
if all_doc_embeddings and query_embeddings_list:
    # Build score matrix
    score_matrix = np.zeros((len(QUERIES), len(all_doc_embeddings)))
    for qi, q_emb in enumerate(query_embeddings_list):
        for di, d_emb in enumerate(all_doc_embeddings):
            score_matrix[qi, di] = maxsim_score(q_emb, d_emb)

    # Display as HTML table
    max_score = score_matrix.max() if score_matrix.max() > 0 else 1.0
    header = "<tr><th>Query</th>" + "".join(
        f"<th style='font-size:11px; max-width:120px; overflow:hidden'>{lbl}</th>"
        for lbl in all_doc_labels
    ) + "</tr>"

    rows = ""
    for qi, query in enumerate(QUERIES):
        cells = "".join(
            f"<td style='background-color: rgba(0,128,0,{score_matrix[qi, di] / max_score:.2f}); "
            f"text-align:center; color:white; font-weight:bold'>{score_matrix[qi, di]:.2f}</td>"
            for di in range(len(all_doc_embeddings))
        )
        rows += f"<tr><td style='font-size:12px'>{query[:50]}</td>{cells}</tr>"

    display(HTML(f"""
    <h3>MaxSim Score Matrix</h3>
    <table border="1" cellpadding="4" style="border-collapse:collapse; font-size:12px;">
    <thead>{header}</thead>
    <tbody>{rows}</tbody>
    </table>
    <p><em>Darker green = higher relevance score. Scores are MaxSim (sum of per-query-patch max cosine similarities).</em></p>
    """))
else:
    print("No data available for score matrix. Run embedding sections first.")

Query,img: test_png.png,pdf: test_pdf.pdf p.1,pdf: test_pdf.pdf p.2,pdf: test_pdf.pdf p.3,pdf: test_pdf.pdf p.4,pdf: test_pdf.pdf p.5,pdf: test_pdf.pdf p.6,pdf: test_pdf.pdf p.7,pdf: test_pdf.pdf p.8,pdf: test_pdf.pdf p.9,pdf: test_pdf.pdf p.10,pdf: test_pdf.pdf p.11,pdf: test_pdf.pdf p.12,pdf: test_pdf.pdf p.13,pdf: test_pdf.pdf p.14,pdf: test_pdf.pdf p.15,pdf: test_pdf.pdf p.16,pdf: test_pdf.pdf p.17,pdf: test_pdf.pdf p.18,pdf: test_pdf.pdf p.19,pdf: test_pdf.pdf p.20,pdf: test_pdf.pdf p.21,pdf: test_pdf.pdf p.22,pdf: test_pdf.pdf p.23,pdf: test_pdf.pdf p.24,pdf: test_pdf.pdf p.25,pdf: test_pdf.pdf p.26,pdf: test_pdf.pdf p.27,pdf: test_pdf.pdf p.28,pdf: test_pdf.pdf p.29,pdf: test_pdf.pdf p.30,pdf: test_pdf.pdf p.31,pdf: test_pdf.pdf p.32,pdf: test_pdf.pdf p.33,pdf: test_pdf.pdf p.34,pdf: test_pdf.pdf p.35,pdf: test_pdf.pdf p.36,pdf: test_pdf.pdf p.37,pdf: test_pdf.pdf p.38,pdf: test_pdf.pdf p.39,pdf: test_pdf.pdf p.40,pdf: test_pdf.pdf p.41,pdf: test_pdf.pdf p.42,pdf: test_pdf.pdf p.43,pdf: test_pdf.pdf p.44,pdf: test_pdf.pdf p.45,pdf: test_pdf.pdf p.46,pdf: test_pdf.pdf p.47,pdf: test_pdf.pdf p.48,pdf: test_pdf.pdf p.49,pdf: test_pdf.pdf p.50,pdf: test_pdf.pdf p.51,pdf: test_pdf.pdf p.52,pdf: test_pdf.pdf p.53,pdf: test_pdf.pdf p.54,pdf: test_pdf.pdf p.55,pdf: test_pdf.pdf p.56,pdf: test_pdf.pdf p.57,pdf: test_pdf.pdf p.58,pdf: test_pdf.pdf p.59,pdf: test_pdf.pdf p.60,pdf: test_pdf.pdf p.61,pdf: test_pdf.pdf p.62,pdf: test_pdf.pdf p.63,pdf: test_pdf.pdf p.64,pdf: test_pdf.pdf p.65,pdf: test_pdf.pdf p.66,pdf: test_pdf.pdf p.67,pdf: test_pdf.pdf p.68,pdf: test_pdf.pdf p.69,pdf: test_pdf.pdf p.70,pdf: test_pdf.pdf p.71,pdf: test_pdf.pdf p.72,pdf: test_pdf.pdf p.73,pdf: test_pdf.pdf p.74,pdf: test_pdf.pdf p.75,pdf: test_pdf.pdf p.76,pdf: test_pdf.pdf p.77,pdf: test_pdf.pdf p.78,pdf: test_pdf.pdf p.79,pdf: test_pdf.pdf p.80,pdf: test_pdf.pdf p.81,docx: test_docx.docx p.1,docx: test_docx.docx p.2,docx: test_docx.docx p.3,docx: test_docx.docx p.4,docx: test_docx.docx p.5,docx: test_docx.docx p.6,docx: test_docx.docx p.7,docx: test_docx.docx p.8,docx: test_docx.docx p.9,docx: test_docx.docx p.10,docx: test_docx.docx p.11,docx: test_docx.docx p.12,docx: test_docx.docx p.13,docx: test_docx.docx p.14,docx: test_docx.docx p.15,docx: test_docx.docx p.16,docx: test_docx.docx p.17,docx: test_docx.docx p.18,docx: test_docx.docx p.19,docx: test_docx.docx p.20,docx: test_docx.docx p.21,docx: test_docx.docx p.22,docx: test_docx.docx p.23,docx: test_docx.docx p.24,docx: test_docx.docx p.25,docx: test_docx.docx p.26,docx: test_docx.docx p.27,docx: test_docx.docx p.28,docx: test_docx.docx p.29,docx: test_docx.docx p.30,docx: test_docx.docx p.31,docx: test_docx.docx p.32,docx: test_docx.docx p.33,docx: test_docx.docx p.34,docx: test_docx.docx p.35,docx: test_docx.docx p.36,docx: test_docx.docx p.37,docx: test_docx.docx p.38,docx: test_docx.docx p.39,docx: test_docx.docx p.40,docx: test_docx.docx p.41,docx: test_docx.docx p.42,docx: test_docx.docx p.43,docx: test_docx.docx p.44,docx: test_docx.docx p.45,docx: test_docx.docx p.46,docx: test_docx.docx p.47,docx: test_docx.docx p.48,docx: test_docx.docx p.49,docx: test_docx.docx p.50,docx: test_docx.docx p.51,docx: test_docx.docx p.52,docx: test_docx.docx p.53,docx: test_docx.docx p.54,docx: test_docx.docx p.55,docx: test_docx.docx p.56,docx: test_docx.docx p.57,docx: test_docx.docx p.58,docx: test_docx.docx p.59,docx: test_docx.docx p.60,docx: test_docx.docx p.61,docx: test_docx.docx p.62,docx: test_docx.docx p.63,docx: test_docx.docx p.64,docx: test_docx.docx p.65,docx: test_docx.docx p.66,docx: test_docx.docx p.67,docx: test_docx.docx p.68,docx: test_docx.docx p.69,docx: test_docx.docx p.70,docx: test_docx.docx p.71,docx: test_docx.docx p.72,docx: test_docx.docx p.73,docx: test_docx.docx p.74,docx: test_docx.docx p.75,docx: test_docx.docx p.76,docx: test_docx.docx p.77,docx: test_docx.docx p.78,docx: test_docx.docx p.79,docx: test_docx.docx p.80,docx: test_docx.docx p.81,docx: test_docx.docx p.82

## Results Export

Export embedding metadata (shapes, patch counts, norms) to JSON. The actual embedding
vectors are large (hundreds of KB per page), so the JSON contains only **metadata** —
use the numpy arrays in memory for further processing, or serialize them separately
with `np.save()` if needed.

Output folder structure:

```
embedding_output/
    <image_stem>/
        <image_stem>.png                    # copy of original image
        <image_stem>_embedding_meta.json    # embedding metadata
    <pdf_stem>/
        page_001.png                        # rasterized page images
        page_002.png
        <pdf_stem>_embedding_meta.json      # per-page embedding metadata
    <docx_stem>/
        <docx_stem>.pdf                     # intermediate PDF from LibreOffice
        page_001.png
        <docx_stem>_embedding_meta.json     # per-page embedding metadata
```

In [23]:
def export_embedding_results(
    embeddings: list[np.ndarray | None],
    source_file: str,
    output_dir: Path | None = None,
) -> Path:
    """
    Export embedding metadata to JSON.

    Saves patch counts, dimensions, and norm statistics per page.
    Actual embedding vectors are not saved (too large for JSON).
    """
    stem = Path(source_file).stem
    if output_dir is None:
        output_dir = OUTPUT_DIR / stem
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{stem}_embedding_meta.json"

    export_data = {
        "source_file": str(source_file),
        "model": MODEL_ID,
        "embedding_dim": EMBEDDING_DIM,
        "total_pages": len(embeddings),
        "pages": [],
    }

    for i, emb in enumerate(embeddings, 1):
        if emb is not None:
            arr = np.asarray(emb, dtype=np.float32)
            norms = np.linalg.norm(arr, axis=1)
            page_data = {
                "page_number": i,
                "num_patches": arr.shape[0],
                "embedding_dim": arr.shape[1],
                "status": "ok",
                "patch_norm_mean": round(float(norms.mean()), 4),
                "patch_norm_std": round(float(norms.std()), 4),
                "memory_kb": round(arr.nbytes / 1024, 1),
            }
        else:
            page_data = {
                "page_number": i,
                "num_patches": 0,
                "embedding_dim": 0,
                "status": "failed",
            }
        export_data["pages"].append(page_data)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(export_data, f, indent=2, ensure_ascii=False)

    print(f"Results exported to: {output_path}")
    print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")
    return output_path


# Uncomment the lines below to export results.
# The JSON file is saved into the same per-document subfolder.

# -- For single image results:
export_embedding_results([image_embedding], IMAGE_PATH, output_dir=img_output_dir)

# -- For PDF results:
export_embedding_results(pdf_embeddings, PDF_PATH, output_dir=pdf_output_dir)

# -- For DOCX results:
export_embedding_results(docx_embeddings, DOCX_PATH, output_dir=docx_output_dir)

Results exported to: embedding_output/test_png/test_png_embedding_meta.json
  File size: 0.3 KB
Results exported to: embedding_output/test_pdf/test_pdf_embedding_meta.json
  File size: 16.0 KB
Results exported to: embedding_output/test_docx/test_docx_embedding_meta.json
  File size: 43.2 KB


PosixPath('embedding_output/test_docx/test_docx_embedding_meta.json')

## Evaluation & Observability

After running the embedding and retrieval sections, open the Opik dashboard at
`http://localhost:5173` to see:

- All embedding traces with token counts and latency
- Per-request metadata (number of images, patches per image, model info)
- Batch vs single-page request performance comparison

For quantitative retrieval evaluation, use a labeled query-document relevance dataset
and compute standard IR metrics (MRR, Recall@K, NDCG). The `rank_documents()` function
above returns ranked results that can be compared against ground truth.

In [24]:
# ── Flush Opik traces and close HTTP client ────────────────────
# Critical for notebooks: without flush(), pending spans may be lost
# because the background sender thread may not finish before the kernel stops.
opik_client.flush()

# Close the shared HTTP client (releases connection pool)
if "vllm_http_client" in dir() and vllm_http_client is not None:
    vllm_http_client.close()

# NOTE: Update this URL if you switch between Cloud and self-hosted Opik.
# Cloud:       https://www.comet.com/opik
# Self-hosted: http://localhost:5173
print("All Opik traces flushed. View results at https://www.comet.com/opik")

All Opik traces flushed. View results at https://www.comet.com/opik
